In [1]:
from collections.abc import Callable, Sequence
from copy import deepcopy
from functools import partial
from random import Random
from statistics import median
from timeit import repeat
from typing import Any

In [2]:
orders = [("A", 2), ("B", 1), ("C", 0), ("D", 1)]
EXTRA_PRIORITIES = (1, 0, 2)
own_orders = [("A", 2), ("B", 1), ("C", 2), ("D", 0), ("E", 1), ("F", 3)]
SORT_SIZES = (40, 80, 160)
BENCHMARK_NUMBER = 5
BENCHMARK_REPEATS = 5
MERGE_LEFT = [("A", 0), ("B", 2), ("C", 2)]
MERGE_RIGHT = [("D", 1), ("E", 2), ("F", 3)]

# ДЗ №3. Сортировки

Курс «Алгоритмы и структуры данных»

**Фамилия, группа:** Ибрагимов, МПМ-26-1

## Как работать с этим ноутбуком

- Дважды щёлкните по первой ячейке, впишите фамилию и группу и нажмите Shift+Enter.
- Демо-ячейки уже выполнены. Запускайте их сверху вниз через Shift+Enter.
  Строки с пометкой `# <-- можно менять здесь` подходят для экспериментов.
- В ячейках «Ответ» замените курсивную подсказку своим текстом. В коде заполните места с пометкой `TODO`.
- Для запуска достаточно Python 3.9+ и стандартной библиотеки. В замерах используйте числа со своего компьютера.

## Шпаргалка

| Понятие | Как читать пример |
|---|---|
| Ключ сортировки | У пары (номер, приоритет) сравниваем приоритет, а переносим всю пару. |
| Устойчивость | Если приоритеты совпали, заявки остаются в порядке поступления. |
| Сдвиг | При вставке одна из уже просмотренных записей смещается вправо на одну позицию. |

---

## Задача 1. Кого обслужить первым?

Четыре заявки поступили в порядке A → B → C → D. У каждой есть
приоритет: чем меньше число, тем раньше её нужно обработать.
Посмотрим, какую очередь получит оператор. Потом поменяйте приоритет
заявки C с 2 на 0 и сразу увидите, куда она переместится.

In [3]:
def priority(order: tuple[str, int]) -> int:
    return order[1]


ready = sorted(orders, key=priority)
print("Пришли:    ", orders)
print("Обслужим:  ", ready)
print("Очередь:   ", " → ".join(order[0] for order in ready))

Пришли:     [('A', 2), ('B', 1), ('C', 0), ('D', 1)]
Обслужим:   [('C', 0), ('B', 1), ('D', 1), ('A', 2)]
Очередь:    C → B → D → A


В исходном примере получилось **B → D → A → C**. B и D срочнее остальных, но B пришла
раньше D и остаётся впереди. То же самое происходит с A и C.

Разберём запись, чтобы можно было её менять.

- `orders` — список. Квадратные скобки объединяют несколько заявок.
- `("A", 2)` — кортеж, то есть одна запись из двух полей:
  номер заявки и приоритет. Поле `[0]` — номер, `[1]` — приоритет.
- `def priority(order)` задаёт функцию; `return order[1]` возвращает
  только число, по которому нужно сравнивать заявки. Это **ключ сортировки**.
- `sorted(..., key=priority)` создаёт новый упорядоченный список.
  Исходный `orders` остаётся прежним. `for order in ready` проходит
  по записям по очереди; последняя строка только красиво печатает номера.

Сохранение порядка заявок с одинаковым ключом называется
**устойчивостью**. Здесь это понятное правило сервиса: при равном
приоритете раньше обслуживаем того, кто раньше обратился.

### Измените очередь и проверьте ожидание

Сначала мысленно поставьте заявку E на нужное место. Затем запустите
ячейку. Поменяйте её приоритет на 0 или 2 и запустите ещё раз.
При одинаковом приоритете E должна остаться после более ранних заявок.

In [4]:
for extra_priority in EXTRA_PRIORITIES:
    extra_order = ("E", extra_priority)
    changed_orders = orders + [extra_order]
    changed_result = sorted(changed_orders, key=priority)
    print("Приоритет E:", extra_priority, "очередь:", changed_result)
    equal_names = [name for name, value in changed_result if value == extra_priority]
    assert equal_names[-1] == "E"

Приоритет E: 1 очередь: [('C', 0), ('B', 1), ('D', 1), ('E', 1), ('A', 2)]
Приоритет E: 0 очередь: [('C', 0), ('E', 0), ('B', 1), ('D', 1), ('A', 2)]
Приоритет E: 2 очередь: [('C', 0), ('B', 1), ('D', 1), ('A', 2), ('E', 2)]


### Ответ (задача 1)

Изменил приоритет C с 2 на 0: ожидаю и получаю C → B → D → A.
Добавил E и проверил три приоритета: при 1 очередь C → B → D → E → A;
при 0 — C → E → B → D → A; при 2 — C → B → D → A → E.
При равных приоритетах E остаётся после более ранних заявок: сортировка устойчива.
Если убрать key, кортежи сравниваются сначала по номеру заявки, и срочность не учитывается.

### Неудачный вариант: упорядочить не то поле

Что произойдёт, если убрать `key`? Python умеет сравнивать кортежи,
но начинает с **первого** поля. В нашем формате это буква, а не приоритет.
Программа не падает — она просто решает другую задачу. Для этого
контрпримера возьмём фиксированные записи, независимо от ваших изменений.

In [5]:
counterexample_orders = [("A", 2), ("B", 1), ("C", 2), ("D", 1)]
wrong_queue = sorted(counterexample_orders)
print("Без key:    ", wrong_queue)
print("По приоритету:", sorted(counterexample_orders, key=priority))
assert wrong_queue != sorted(counterexample_orders, key=priority)
print("Контрпример: A с приоритетом 2 оказалась раньше B с приоритетом 1.")

Без key:     [('A', 2), ('B', 1), ('C', 2), ('D', 1)]
По приоритету: [('B', 1), ('D', 1), ('A', 2), ('C', 2)]
Контрпример: A с приоритетом 2 оказалась раньше B с приоритетом 1.


Сортировка должна сохранить **все записи**, правильно расположить
ключи и, когда это требуется, сохранить порядок равных ключей.
Красивого вывода без ошибки Python для этого недостаточно.

---

## Задача 2. Упорядочите заявки сортировкой вставками

Возьмём сортировку **вставками**. Представьте карточки на столе:
слева уже готовая упорядоченная часть, справа — ещё не просмотренные.
Берём следующую карточку, сохраняем её отдельно, сдвигаем вправо
только карточки с **большим** приоритетом и ставим свою в пустое место.

| До шага | Что делаем | После шага |
|---|---|---|
| A:3 │ B:1, C:2 | Сохраняем B; сдвигаем A; ставим B | B:1, A:3 │ C:2 |
| B:1, A:3 │ C:2 | Сохраняем C; сдвигаем A; на B останавливаемся | B:1, C:2, A:3 |

Черта отделяет готовую начальную часть, **префикс**. Проследим только
первый шаг в коде: это одна вставка, ещё не алгоритм для всего списка.

In [6]:
cards = [("A", 3), ("B", 1), ("C", 2)]
current = cards[1]  # сохраняем B, чтобы сдвиг её не затёр
cards[1] = cards[0]  # сдвигаем A вправо
cards[0] = current  # возвращаем B в освободившееся место
print("После первой вставки:", cards)

После первой вставки: [('B', 1), ('A', 3), ('C', 2)]


На первом шаге сравнили 3 и 1: одно сравнение, один сдвиг.
На втором сравнили 3 с 2, затем 1 с 2: два сравнения, один сдвиг.
Итого **3 сравнения ключей, 2 сдвига**. Проверку индекса и финальную
постановку сохранённой записи в эти два счётчика не включаем.

Равные ключи не нужно сдвигать. Иначе поздняя заявка окажется перед
ранней: например, A:2, B:2 превратятся в B:2, A:2. На одинаковых
числах перестановка незаметна — поэтому проверяем записи с номерами.

Почему вставки работают: перед шагом готовый префикс уже отсортирован.
Сдвиги освобождают место для следующей записи и не теряют остальные.
После вставки упорядоченный префикс становится на одну запись длиннее.
Так он дорастает до всего списка.

### Реализация на Python

В `insertion_sort` уже готовы циклы, индексы и счётчики. Заполните
только **два места TODO**, заменив `None` на выражения Python.
Функция принимает список и функцию
`key`, возвращает **(новый список, число сравнений, число сдвигов)**.
Не меняйте входные записи и сам входной список. Пустой список тоже допустим.
Ключи — обычные сравнимые числа или строки, без NaN и побочных эффектов.
`sorted` и `.sort()` внутри вашей функции не используйте: здесь вы
собираете механизм сами, а библиотека проверяет его результат.

Что пригодится в Python:

- `result = list(items)` создаёт копию внешнего списка.
- `for i in range(1, len(result))` берёт позиции от 1 до последней.
- `while j >= 0` продолжает просмотр влево, пока индекс допустим.
- `j -= 1` уменьшает индекс; `break` выходит из цикла.
- `key(result[j])` получает ключ левой записи; `key(current)` — текущей.
- `lambda value: value` — короткая функция, возвращающая аргумент.
  Это ключ по умолчанию для обычных чисел. Для заявок передадим `priority`.

В **TODO 1** нужно логическое условие: больше ли ключ левой записи
текущего ключа? При равенстве двигать нельзя. В **TODO 2** нужна
целая левая запись, которую перенесём в позицию `j+1`, а не только её ключ.
Остальная механика уже написана. Строки `return None` ниже проверяют,
остались ли незаполненные места: удалять эти защитные строки не нужно.

### Ответ (задача 2)

Сдвигаю левую запись, только когда key(result[j]) > current_key; при равенстве останавливаюсь.
В освободившееся место переношу всю запись result[j], а не её ключ.
Так сохраняются устойчивость и состав записей, исходный список остаётся неизменным благодаря копии.
Перед каждой вставкой префикс отсортирован; после сдвигов и вставки он увеличивается на один элемент.
Счётчик сравнений учитывает и последнюю неуспешную проверку ключей, если она была.

In [7]:
def insertion_sort(items: Sequence[Any], key: Callable[[Any], Any] = lambda value: value) -> tuple[list[Any], int, int] | None:
    result = list(items)
    comparisons = 0
    shifts = 0
    for i in range(1, len(result)):
        current = result[i]
        current_key = key(current)
        j = i - 1
        while j >= 0:
            comparisons += 1
            move_right = key(result[j]) > current_key
            if move_right is None:
                return None  # диагностика незаполненного места
            if not move_right:
                break
            record_to_move = result[j]
            if record_to_move is None:
                return None  # диагностика незаполненного места
            result[j + 1] = record_to_move
            shifts += 1
            j -= 1
        result[j + 1] = current
    return result, comparisons, shifts

### Проверим функцию

Следующая ячейка готова: редактировать её не нужно. Пока в функции
оставлены `None` в TODO, она пишет **НЕ ВЫПОЛНЕНО**. После заполнения
проверит пустые входы, повторы, счётчики, устойчивость и сохранность данных.
**ПРОЙДЕНО** означает успех этих тестов, а не готовность ваших объяснений.

В тестах встречается и словарь `{"key": -1, "id": "a"}`: это запись
с именованными полями. `record["key"]` читает поле с именем `key`,
а `lambda record: record["key"]` — функция для такого формата.
Это проверяет, что вы сравниваете ключи, а переносите записи целиком.

In [8]:
CHECK_RESULTS = {}


def check_work(name, probe, tests):
    """None means unfinished; an exception means a failed check, not success."""
    try:
        if probe() is None:
            CHECK_RESULTS[name] = "НЕ ВЫПОЛНЕНО"
            print(name + ": НЕ ВЫПОЛНЕНО — заполните TODO.")
            return False
        tests()
    except Exception as error:
        CHECK_RESULTS[name] = "ОШИБКА"
        print(name + ": ОШИБКА — " + type(error).__name__)
        print(str(error)[:160])
        return False
    CHECK_RESULTS[name] = "ПРОЙДЕНО"
    print(name + ": ПРОЙДЕНО — открытые тесты; объяснения отдельно.")
    return True


def test_sort():
    rng = Random(42)
    cases = [[], [3], [1, 2, 3], [3, 2, 1], [2, 2, 2], [0, -1, 4, -1, 0]]
    cases += [[rng.randrange(-5, 6) for _ in range(n)] for n in range(30)]
    for values in cases:
        before = values[:]
        actual = insertion_sort(values)
        assert isinstance(actual, tuple) and len(actual) == 3
        result, comparisons, shifts = actual
        assert type(comparisons) is int and type(shifts) is int
        assert result == sorted(values), (values, result)
        assert values == before, "Изменён входной список"
        assert result is not values, "Нужен новый список"
        inversions = sum(values[i] > values[j] for i in range(len(values)) for j in range(i + 1, len(values)))
        assert shifts == inversions, "Сдвиги должны равняться инверсиям"
        # Независимый подсчёт: большие предыдущие ключи + остановочное
        # сравнение, если среди предыдущих есть ключ <= текущего.
        expected_comparisons = sum(
            sum(previous > values[i] for previous in values[:i]) + int(any(previous <= values[i] for previous in values[:i]))
            for i in range(1, len(values))
        )
        assert comparisons == expected_comparisons, "Неверны сравнения"
    for n in [0, 1, 2, 8, 20]:
        assert insertion_sort(list(range(n)))[1:] == (max(0, n - 1), 0)
        worst = n * (n - 1) // 2
        assert insertion_sort(list(range(n, 0, -1)))[1:] == (worst, worst)
        assert insertion_sort([2] * n)[1:] == (max(0, n - 1), 0)
    tuples = [(2, "a"), (1, "b"), (2, "c"), (1, "d")]
    dictionaries = [{"key": -1, "id": "a"}, {"key": 2, "id": "b"}, {"key": -1, "id": "c"}, {"key": 2, "id": "d"}]
    for records, key in [(tuples, lambda item: item[0]), (dictionaries, lambda item: item["key"]), (["BB", "a", "C", "dd"], len)]:
        before = deepcopy(records)
        expected = sorted(deepcopy(records), key=key)
        result, _, _ = insertion_sort(records, key=key)
        assert result == expected, "Нарушены порядок или состав записей"
        assert records == before
    assert insertion_sort([3, 1, 2])[1:] == (3, 2)


sort_ok = check_work("Сортировка вставками", lambda: insertion_sort([2, 1]), test_sort)

Сортировка вставками: ПРОЙДЕНО — открытые тесты; объяснения отдельно.


---

## Задача 3. Составьте свою очередь

Придумайте 4–6 заявок: у двух должен быть одинаковый приоритет, а одна
срочная заявка должна прийти позже остальных. Впишите их вместо `None`.
Функцию `priority` используем прежнюю: номер первым, приоритет вторым.
Перед запуском напишите ожидаемый порядок номеров в следующей заметке.

### Ответ (задача 3)

Моя ситуация — очередь технических заявок: A и C обычные (2), B и E важные (1),
D аварийная (0), F отложенная (3). D поступает четвёртой, но должна обслуживаться первой.
Ожидаю D → B → E → A → C → F: внутри каждой группы равных приоритетов сохраняется порядок поступления.

После запуска порядок совпал с прогнозом; выполнено 9 сравнений ключей
и 6 сдвигов. Проверка подтвердила, что исходная очередь не изменилась.

In [9]:
print("Своя очередь до сортировки:", own_orders)

Своя очередь до сортировки: [('A', 2), ('B', 1), ('C', 2), ('D', 0), ('E', 1), ('F', 3)]


In [10]:
if own_orders is None or not sort_ok:
    own_result = None
    print("Своя очередь: НЕ ВЫПОЛНЕНО — нужны функция и свои заявки.")
else:
    before = deepcopy(own_orders)
    own_result, comparisons, shifts = insertion_sort(own_orders, key=priority)
    assert own_result == sorted(before, key=priority)
    assert own_orders == before
    print("Пришли:    ", own_orders)
    print("Обслужим:  ", own_result)
    print("Сравнений:", comparisons, "сдвигов:", shifts)

Пришли:     [('A', 2), ('B', 1), ('C', 2), ('D', 0), ('E', 1), ('F', 3)]
Обслужим:   [('D', 0), ('B', 1), ('E', 1), ('A', 2), ('C', 2), ('F', 3)]
Сравнений: 9 сдвигов: 6


---

## Задача 4. Сравните работу вставок на разных входах

Теперь готовый стенд подаст функции четыре вида входа: `ordered` —
готовый порядок, `reversed` — обратный, `shuffled` — перестановка,
`duplicates` — значения 0–3 с повторами. Сначала мысленно сравните
готовую и обратную очереди: где нужны сдвиги?

Сравнения и сдвиги показывают работу алгоритма. Время в микросекундах
лишь дополняет их: в вашей функции есть цикл Python и счётчики,
а `sorted` реализован иначе. Подготовка входа не измеряется.
Каждый вызов получает тот же неизменённый список; обе функции
создают новый. Время — медиана 5 повторов по 5 запусков.

In [11]:
def sort_inputs(n):
    rng = Random(42)
    random_order = list(range(n))
    rng.shuffle(random_order)
    return {
        "ordered": list(range(n)),
        "reversed": list(range(n, 0, -1)),
        "shuffled": random_order,
        "duplicates": [rng.randrange(4) for _ in range(n)],
    }


def time_sort(action):
    action()
    return median(repeat(action, number=BENCHMARK_NUMBER, repeat=BENCHMARK_REPEATS)) / BENCHMARK_NUMBER


def sort_experiment():
    rows = []
    print(" n  вход        сравн. сдвиги  вставки мкс  sorted мкс")
    for n in SORT_SIZES:
        for name, data in sort_inputs(n).items():
            result, comparisons, shifts = insertion_sort(data)
            assert result == sorted(data)
            custom = time_sort(partial(insertion_sort, data))
            builtin = time_sort(partial(sorted, data))
            rows.append((n, name, comparisons, shifts, custom, builtin))
            print(f"{n:3} {name:10} {comparisons:6} {shifts:6} {custom * 1e6:12.2f} {builtin * 1e6:11.2f}")
    return rows


if sort_ok:
    sort_results = sort_experiment()
else:
    sort_results = None
    print("Эксперимент: НЕ ВЫПОЛНЕНО — сначала исправьте сортировку.")

 n  вход        сравн. сдвиги  вставки мкс  sorted мкс
 40 ordered        39      0         4.96        0.42
 40 reversed      780    780        89.06        0.76
 40 shuffled      467    431        64.65        1.05
 40 duplicates    337    299        36.18        1.00
 80 ordered        79      0         9.53        0.67
 80 reversed     3160   3160       381.60        0.90
 80 shuffled     1733   1658       195.28        2.44
 80 duplicates   1407   1329       157.90        2.21
160 ordered       159      0        18.88        1.32
160 reversed    12720  12720      1462.87        1.43
160 shuffled     6793   6637       772.06        5.93
160 duplicates   4817   4659       513.78        4.52


На готовом порядке вставки делают n−1 сравнение и ни одного сдвига.
На обратном из различных чисел — n(n−1)/2 сравнений и сдвигов.
Поэтому работа растёт примерно вдвое и вчетверо соответственно
при удвоении n: это линейный Θ(n) и квадратичный Θ(n²) рост.
Θ обозначает точный порядок роста с точностью до постоянного множителя,
O — верхнюю оценку роста. Это число операций, а не миллисекунды.
Копия результата занимает O(n) дополнительной памяти.

Сверьте эти ожидания со строками для 80 и 160 элементов. Нулевые сдвиги
не означают нулевую работу. Близкие времена могут меняться при повторе;
по трём замерам нельзя доказать асимптотику или универсальный выигрыш.

### Что показать в результате

Сохраните этот ноутбук с рабочей функцией, своей очередью и её выводом.
Добавьте **три коротких вывода**:

1. Кто в вашей очереди оказался первым и почему равные приоритеты
   сохранили или не сохранили порядок?
2. Как изменились сравнения/сдвиги от 80 к 160 на двух разных входах?
3. Когда вы выбрали бы вставки, а когда готовый `sorted`?

### Ответ (задача 4)

1. Первой стала аварийная D. Равные приоритеты сохраняют порядок, потому что сдвигаются только строго большие ключи.
2. Для упорядоченного входа при переходе от 80 к 160 элементам сравнений 79 → 159, сдвигов 0 → 0.
   Для обратного входа с различными числами сравнений и сдвигов 3160 → 12720: рост примерно в четыре раза.
   Это согласуется с Θ(n) в лучшем случае и Θ(n²) в худшем; время само по себе не доказывает асимптотику.
3. Вставки подходят для маленьких или почти упорядоченных наборов: число сдвигов равно числу инверсий.
   Для общего случая на Python выберу sorted, учитывая его реализацию и гарантии устойчивости.
   Численные результаты времени приведены в сохранённом выводе стенда; копирование результата требует O(n) памяти.

Время сортировки вставками в данном запуске:

| Вход | 80 элементов, мкс | 160 элементов, мкс |
| --- | --- | --- |
| Упорядоченный | 9.53 | 18.88 |
| Обратный | 381.60 | 1462.87 |

Встроенный sorted оказался быстрее нашей реализации во всех двенадцати измеренных случаях.
Сравниваются разные реализации, включая накладные расходы Python и наших счётчиков,
поэтому это не доказательство универсального превосходства одного абстрактного алгоритма.

---

## Задача 5 (со звёздочкой). Слияние очередей

По желанию объедините две уже
упорядоченные очереди заявок, сохранив порядок равных приоритетов.
При равенстве берите запись из левой очереди первой. Проверьте случай,
когда одна очередь пуста. Полная сортировка слиянием не требуется.

[Описание sorted и устойчивости](https://docs.python.org/3/howto/sorting.html).

In [12]:
def merge_queues(left: Sequence[tuple[str, int]], right: Sequence[tuple[str, int]]) -> list[tuple[str, int]]:
    merged = []
    left_index = right_index = 0
    while left_index < len(left) and right_index < len(right):
        if priority(left[left_index]) <= priority(right[right_index]):
            merged.append(left[left_index])
            left_index += 1
        else:
            merged.append(right[right_index])
            right_index += 1
    merged.extend(left[left_index:])
    merged.extend(right[right_index:])
    return merged


def test_merge_queues() -> None:
    cases = [([], []), ([], [("A", 1)]), ([("A", 1)], []), ([("A", 1), ("B", 2)], [("C", 1), ("D", 2)]), ([("A", 0), ("B", 0)], [("C", 0)])]
    rng = Random(42)
    for size in range(25):
        left = sorted([(f"L{index}", rng.randrange(4)) for index in range(size)], key=priority)
        right = sorted([(f"R{index}", rng.randrange(4)) for index in range(size)], key=priority)
        cases.append((left, right))
    for left, right in cases:
        before = deepcopy((left, right))
        result = merge_queues(left, right)
        assert result == sorted(left + right, key=priority)
        assert (left, right) == before
        assert result is not left and result is not right


test_merge_queues()
print("Слияние:", merge_queues(MERGE_LEFT, MERGE_RIGHT))
print("Проверки слияния: ПРОЙДЕНО")

Слияние: [('A', 0), ('D', 1), ('B', 2), ('C', 2), ('E', 2), ('F', 3)]
Проверки слияния: ПРОЙДЕНО


### Ответ (задача 5)

На каждом шаге переношу меньший из первых оставшихся приоритетов; при равенстве беру левую запись.
Так сохраняется порядок внутри каждой очереди и левый порядок при равенстве между очередями.
В примере результат A → D → B → C → E → F. Пустая очередь обрабатывается дописыванием второй.
Время O(n + m), дополнительная память O(n + m) для нового списка; входные очереди не изменяются.


---

## Сдача

- В первой ячейке указаны ваши фамилия и группа.
- Заполнены ответы к задачам, код функции и данные для собственного примера.
- Ядро перезапущено, ячейки выполнены сверху вниз. Выводы с вашими результатами сохранены в файле.
- Сохраните и сдайте заполненный файл в формате `.ipynb`.

Если проверка не проходит, рядом с ней укажите входные данные, ожидаемый результат и фактический вывод.